# DeFi-LSA

This notebook clones a GitHub repository, loads required credentials from Kaggle Secrets, installs the package, writes runtime-only source configuration, runs tests, performs connectivity checks, and launches D2 collection.

## Required Kaggle Secrets
- `GITHUB_REPO_URL`: HTTPS URL of the public repository, or a URL that does not embed credentials.
- `GITHUB_TOKEN`: optional, only for a private repository.
- `ETH_RPC_URL`: Ethereum mainnet JSON-RPC endpoint.
- `ETHERSCAN_API_KEY`: Etherscan API key.

DefiLlama historical pricing requires no API key. The collector uses the verified exploit timestamp minus one second for the fixed USD anchor; if no valid pre-event quote is available, the price-anchor report identifies the token and exact fallback timestamp.

Enable Internet in the Kaggle notebook settings before running.

In [1]:
from pathlib import Path
import os, json, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def get_secret(name, required=False):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle Secret: {name}")
    return value

repo_url = f'https://{get_secret("GITHUB_TOKEN")}@github.com/XQuyTran/DeFi-LSA.git'
repo_branch = 'feature/epic-revised'
os.environ["ETH_RPC_URL"] = get_secret("ETH_RPC_URL", required=True)
os.environ["ETHERSCAN_API_KEY"] = get_secret("ETHERSCAN_API_KEY", required=True)

WORK = Path("/kaggle/working")
REPO = WORK / "defi-lsa"
OUTPUT = WORK / "defi-lsa-output"
restored_candidates = [
    WORK / "defi-lsa-output",
    WORK / "input" / "defi-lsa-output",
]
for candidate in restored_candidates:
    if any((candidate / marker).exists() for marker in ("data", "checkpoints", "reports")):
        OUTPUT = candidate
        break
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Runtime root selected: {OUTPUT}")
print("Secrets loaded; values are not displayed.")

Runtime root selected: /kaggle/working/defi-lsa-output
Secrets loaded; values are not displayed.


In [2]:
import urllib.request
urllib.request.urlopen("https://github.com", timeout=15).read(64)
print("Internet connectivity: OK")


Internet connectivity: OK


In [3]:
import subprocess, os, base64
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", 
                    "--depth", "1", 
                    '--branch', repo_branch, repo_url, str(REPO)], check=True, env=os.environ)
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


Cloning into '/kaggle/working/defi-lsa'...


a07b212aadfb297527af35e0c8780945e41b34f5


In [4]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[dev,parquet,aws]"], check=True)
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
import defi_lsa
print(f"Package installation: OK ({defi_lsa.__file__})")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
boto3 1.44.4 requires botocore<1.45.0,>=1.44.4, but you have botocore 1.43.117 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.


Package installation: OK (/kaggle/working/defi-lsa/src/defi_lsa/__init__.py)


In [5]:
runtime_config = json.loads((REPO / "configs" / "sources.json").read_text(encoding="utf-8"))
runtime_config["coingecko"]["enabled"] = False
runtime_config["defillama"] = {
  "enabled": True,
  "base_url": "https://coins.llama.fi/prices"
}
runtime_config.setdefault("source_priority", {})["prices"] = ["defillama", "chainlink", "onchain_pool"]
runtime_config["defillama_api_url"] = "https://coins.llama.fi/prices"
runtime_config["gdelt"]["enabled"] = True
runtime_config["ethereum"]["rpc"]["request_timeout_seconds"] = 30
runtime_config["ethereum"]["rpc"]["maximum_attempts"] = 5
runtime_config["ethereum"]["rpc"]["requests_per_second"] = 2.0
runtime_config["collection"]["initial_log_chunk_size"] = 2000
runtime_config["collection"]["minimum_log_chunk_size"] = 10
runtime_config["collection"]["maximum_log_chunk_size"] = 20000
config_path = OUTPUT / "runtime_sources.json"
config_path.write_text(json.dumps(runtime_config, indent=2), encoding="utf-8")
print(f"Secret-free runtime configuration written to {config_path}")

Secret-free runtime configuration written to /kaggle/working/defi-lsa-output/runtime_sources.json


In [6]:
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
from defi_lsa.config import load_source_config
from defi_lsa.runtime import preflight_internet, preflight_output_root

runtime_source_config = load_source_config(config_path)
if not runtime_source_config.source_enabled("defillama"):
    raise RuntimeError("DefiLlama must be enabled for the fixed price anchor")
if runtime_source_config.source_enabled("coingecko"):
    raise RuntimeError("CoinGecko is a legacy source and must remain disabled")
preflight = preflight_output_root(OUTPUT, minimum_free_bytes=1_000_000_000)
if not preflight["passed"]:
    raise RuntimeError(f"Runtime preflight failed: {preflight}")
defillama_probe_url = (
    f"{runtime_source_config.defillama_api_url.rstrip('/')}/current/ethereum:"
    "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2"
)
internet = preflight_internet(defillama_probe_url)
if not internet["reachable"]:
    raise RuntimeError(f"DefiLlama API preflight failed: {internet}")
print({"defillama_enabled": True, "coingecko_enabled": False, "output_preflight": preflight, "defillama_api_preflight": internet})

{'defillama_enabled': True, 'coingecko_enabled': False, 'output_preflight': {'output_root': '/kaggle/working/defi-lsa-output', 'restored_output': False, 'free_bytes': 20937273344, 'minimum_free_bytes': 1000000000, 'disk_space_ok': True, 'writable': True, 'passed': True}, 'defillama_api_preflight': {'reachable': True, 'status': 200}}


In [7]:
# Verify both the chain RPC and DefiLlama source endpoints.
for source_name in ("rpc", "defillama"):
    subprocess.run([
        sys.executable, "-m", "defi_lsa", "connectivity",
        "--event", "euler",
        "--config", str(config_path),
        "--source", source_name
    ], cwd=REPO, check=True)

{"details": {"ethereum_rpc": {}}, "event": "euler", "run_id": "ac140eadaff34d589283f23ed265e206", "statuses": {"ethereum_rpc": "reachable"}}
{"details": {"defillama": {}}, "event": "euler", "run_id": "1dcb1d14c1564c429ee0973abee8bd04", "statuses": {"defillama": "reachable"}}


## Smoke collection
Run a small real-data collection before a full event window. The CLI must be wired by Epic E3. If the current repository only writes a summary, stop and implement E3 rather than interpreting the command as a successful collection.


In [8]:
SMOKE_RANGES = {
    "euler": {
        "one_block": (16818024, 16818024),
        "100_blocks": (16817925, 16818024),
    },
    "kyber": {
        "one_block": (18630392, 18630392),
        "100_blocks": (18630293, 18630392),
    },
}
SMOKE_AUDITS = {}
for event, ranges in SMOKE_RANGES.items():
    for label, (start_block, end_block) in ranges.items():
        smoke_output = OUTPUT / "smoke" / f"{event}_{label}"
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "collect",
            "--event", event,
            "--stage", "all",
            "--source", "auto",
            "--start-block", str(start_block),
            "--end-block", str(end_block),
            "--config", str(config_path),
            "--output", str(smoke_output),
            "--resume"
        ], cwd=REPO, check=True)
        audit_event = "kyberswap" if event == "kyber" else event
        audit_result = subprocess.run([
            sys.executable, "-m", "defi_lsa", "audit",
            "--event", audit_event,
            "--stage", "d2",
            "--output", str(smoke_output),
            "--data-dir", str(smoke_output),
            "--strict"
        ], cwd=REPO, capture_output=True, text=True)
        audit_candidates = sorted(smoke_output.rglob("d2_collection_audit.json"))
        if not audit_candidates:
            raise RuntimeError(
                f"Audit report was not created for {event}/{label}; "
                f"returncode={audit_result.returncode}; "
                f"stdout={audit_result.stdout}; stderr={audit_result.stderr}"
            )
        audit_path = audit_candidates[0]
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        status = audit.get("status")
        if status == "NO_GO":
            raise RuntimeError(f"Smoke audit failed for {event}/{label}: {audit}")
        if status == "CONDITIONAL_GO":
            print(
                f"Approved CONDITIONAL_GO for {event}/{label}; D2 hourly price coverage "
                f"may remain conditional while the fixed DefiLlama anchor is recorded separately."
            )
        elif status != "GO":
            raise RuntimeError(f"Unexpected smoke audit status for {event}/{label}: {status}")
        if audit_result.returncode not in (0, 2):
            raise RuntimeError(f"Smoke audit subprocess failed for {event}/{label}: {audit_result.stdout}")
        SMOKE_AUDITS[event, label] = audit_path

{"records_skipped": 0, "records_written": 330, "run_id": "a725c2a33c804100a0ec2b92587b2c1f", "status": "completed"}
Approved CONDITIONAL_GO for euler/one_block; D2 hourly price coverage may remain conditional while the fixed DefiLlama anchor is recorded separately.
{"records_skipped": 2, "records_written": 5145, "run_id": "07f4f9585bcb4eb0bb4def31bcfdf5fa", "status": "completed"}
Approved CONDITIONAL_GO for euler/100_blocks; D2 hourly price coverage may remain conditional while the fixed DefiLlama anchor is recorded separately.
{"records_skipped": 1, "records_written": 816, "run_id": "42654e95cc1c43df920f99db043133e9", "status": "completed"}
Approved CONDITIONAL_GO for kyber/one_block; D2 hourly price coverage may remain conditional while the fixed DefiLlama anchor is recorded separately.
{"records_skipped": 1, "records_written": 7435, "run_id": "102c5bfe7afe4086822f852d2019157f", "status": "completed"}
Approved CONDITIONAL_GO for kyber/100_blocks; D2 hourly price coverage may remain c

In [9]:
for (event, label), audit_path in sorted(SMOKE_AUDITS.items()):
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    print({
        "event": event,
        "label": label,
        "status": audit.get("status"),
        "audit_path": str(audit_path),
        "effective_range": audit.get("effective_range"),
    })

{'event': 'euler', 'label': '100_blocks', 'status': 'CONDITIONAL_GO', 'audit_path': '/kaggle/working/defi-lsa-output/smoke/euler_100_blocks/euler_2023-03-13/d2_collection_audit.json', 'effective_range': {'end_block': 16845224, 'overridden': False, 'registry_end_block': 16845224, 'registry_start_block': 16773224, 'start_block': 16773224}}
{'event': 'euler', 'label': 'one_block', 'status': 'CONDITIONAL_GO', 'audit_path': '/kaggle/working/defi-lsa-output/smoke/euler_one_block/euler_2023-03-13/d2_collection_audit.json', 'effective_range': {'end_block': 16845224, 'overridden': False, 'registry_end_block': 16845224, 'registry_start_block': 16773224, 'start_block': 16773224}}
{'event': 'kyber', 'label': '100_blocks', 'status': 'CONDITIONAL_GO', 'audit_path': '/kaggle/working/defi-lsa-output/smoke/kyber_100_blocks/kyberswap_elastic_2023-11-22/d2_collection_audit.json', 'effective_range': {'end_block': 18630392, 'overridden': False, 'registry_end_block': 18630392, 'registry_start_block': 186303

## Full event runs
Uncomment only after the smoke audit reports `GO` or an approved `CONDITIONAL_GO`, and after Euler/Curve/Kyber registries contain complete windows, incident legs, token mappings, and public-source queries.


In [10]:
EVENTS = ["euler", "kyber"]
RUN_FULL = True  # Set to True to run full collection and audit after smoke tests
if RUN_FULL:
    for event in EVENTS:
        smoke_audits = [
            path for (audit_event, _label), path in SMOKE_AUDITS.items()
            if audit_event == event
        ]
        if not smoke_audits:
            raise RuntimeError(f"Missing smoke audits for {event}")
        approved_statuses = {"GO", "CONDITIONAL_GO"}
        for audit_path in smoke_audits:
            audit = json.loads(audit_path.read_text(encoding="utf-8"))
            if audit.get("status") not in approved_statuses:
                raise RuntimeError(f"Full run blocked for {event}: {audit.get('status')}")
        event_output = OUTPUT / "full" / event
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "collect",
            "--event", event,
            "--stage", "all",
            "--source", "auto",
            "--config", str(config_path),
            "--output", str(event_output),
            "--resume"
        ], cwd=REPO, check=True)
        audit_result = subprocess.run([
            sys.executable, "-m", "defi_lsa", "audit",
            "--event", event,
            "--stage", "d2",
            "--output", str(event_output),
            "--data-dir", str(event_output),
            "--strict"
        ], cwd=REPO, capture_output=True, text=True)
        audit_candidates = sorted(event_output.rglob("d2_collection_audit.json"))
        if not audit_candidates:
            raise RuntimeError(
                f"Full audit report was not created for {event}; "
                f"returncode={audit_result.returncode}; "
                f"stdout={audit_result.stdout}; stderr={audit_result.stderr}"
            )
        audit_path = audit_candidates[0]
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        status = audit.get("status")
        if status == "NO_GO":
            raise RuntimeError(
                f"Full audit failed for {event}: status=NO_GO; "
                f"corrective_actions={audit.get('corrective_actions')}; "
                f"report={audit_path}"
            )
        if status == "CONDITIONAL_GO":
            print(
                f"Approved CONDITIONAL_GO for full {event} run; inspect the price anchor "
                f"report separately from full-window hourly price coverage: {audit_path}"
            )
        elif status != "GO":
            raise RuntimeError(
                f"Unexpected full audit status for {event}: {status}; report={audit_path}"
            )
        if audit_result.returncode not in (0, 2):
            raise RuntimeError(
                f"Full audit subprocess failed for {event}: "
                f"returncode={audit_result.returncode}; "
                f"stdout={audit_result.stdout}; stderr={audit_result.stderr}"
            )

{"records_skipped": 2, "records_written": 3690720, "run_id": "2fe7a6e3760745b896f5bccd1dafd179", "status": "completed"}
Approved CONDITIONAL_GO for full euler run; inspect the price anchor report separately from full-window hourly price coverage: /kaggle/working/defi-lsa-output/full/euler/euler_2023-03-13/d2_collection_audit.json
{"records_skipped": 1, "records_written": 816, "run_id": "ac31314f653547c982e25540b4b54f04", "status": "completed"}
Approved CONDITIONAL_GO for full kyber run; inspect the price anchor report separately from full-window hourly price coverage: /kaggle/working/defi-lsa-output/full/kyber/kyberswap_elastic_2023-11-22/d2_collection_audit.json


## D3 decoding and classification

Set `RUN_D3 = True` only after the Euler and Kyber D2 audits have completed. D3 runs against isolated event roots because its canonical Parquet filenames are shared within an output root. Curve remains outside this run.

In [11]:
RUN_D3 = True
D3_EVENTS = {
    "euler": "euler_2023-03-13",
    "kyber": "kyberswap_elastic_2023-11-22",
}
D3_OUTPUTS = [
    "decoded_events.parquet",
    "decoded_calls.parquet",
    "flow_legs.parquet",
    "classified_transactions.parquet",
    "actor_roles.parquet",
    "incident_legs_resolved.parquet",
    "balance_inputs.parquet",
]
D3_REPORTS = ["unknown_event_report.json", "d3_gate_report.json"]

if RUN_D3:
    for event_name, event_id in D3_EVENTS.items():
        event_output = OUTPUT / "full" / event_name
        audit_candidates = sorted(event_output.rglob("d2_collection_audit.json"))
        if not audit_candidates:
            raise RuntimeError(f"Missing D2 audit for {event_name}: {event_output}")
        audit_path = audit_candidates[0]
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        if audit.get("status") not in {"GO", "CONDITIONAL_GO"}:
            raise RuntimeError(f"D3 blocked for {event_name}: {audit.get('status')}")
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "d3",
            "--event", event_id,
            "--output", str(event_output),
            "--events", str(REPO / "configs" / "events.json"),
            "--incident-legs", str(REPO / "configs" / "incident_legs.json"),
        ], cwd=REPO, check=True)
        d3_report_path = event_output / "reports" / event_id / "d3_gate_report.json"
        if not d3_report_path.exists():
            raise RuntimeError(f"D3 gate report was not created for {event_name}")
        missing_outputs = [
            name for name in D3_OUTPUTS
            if not (event_output / "data" / "decoded" / name).exists()
        ]
        missing_reports = [
            name for name in D3_REPORTS
            if not (event_output / "reports" / event_id / name).exists()
        ]
        if missing_outputs or missing_reports:
            raise RuntimeError({
                "event": event_name,
                "missing_outputs": missing_outputs,
                "missing_reports": missing_reports,
            })
        print({
            "event": event_name,
            "d2_status": audit.get("status"),
            "d3_report": str(d3_report_path),
        })

{"attribution_row_count": 5864, "corrective_actions": [], "d2_gate_status": "CONDITIONAL_GO", "decoded_call_count": 257, "decoded_event_count": 15981, "event_id": "euler_2023-03-13", "excluded_unrelated_token_transfer_count": 3508805, "exploit_transaction_count": 1, "flow_label_counts": {"attacker_drain": 4, "user_net_flow": 9090}, "flow_leg_count": 9094, "flow_volume_audit_by_token": [{"passed": true, "threshold": 0.05, "token_address": "0x2260fac5e5542a773aa44fbcfedf7c193bc2c599", "total_value_wei": "2187320746647", "unknown_flow_share": 0.0, "unknown_value_wei": "0"}, {"passed": true, "threshold": 0.05, "token_address": "0x6b175474e89094c44da98b954eedeac495271d0f", "total_value_wei": "704627774377901341181384007", "unknown_flow_share": 0.0, "unknown_value_wei": "0"}, {"passed": true, "threshold": 0.05, "token_address": "0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0", "total_value_wei": "1423403582846417998885508", "unknown_flow_share": 0.0, "unknown_value_wei": "0"}, {"passed": true, "

In [12]:
# Sanity-check generated artifacts without exposing secrets.
for path in sorted(OUTPUT.rglob("*")):
    if path.is_file() and path != config_path:
        print(path.relative_to(OUTPUT), path.stat().st_size)


full/euler/data/decoded/actor_roles.parquet 234225
full/euler/data/decoded/balance_inputs.parquet 347043
full/euler/data/decoded/classified_transactions.parquet 253500
full/euler/data/decoded/decoded_calls.parquet 133480
full/euler/data/decoded/decoded_events.parquet 2406133
full/euler/data/decoded/flow_legs.parquet 572665
full/euler/data/decoded/incident_legs_resolved.parquet 5387
full/euler/data/raw/normalized/blocks/euler_2023-03-13.jsonl 24986332
full/euler/data/raw/normalized/contracts/euler_2023-03-13.jsonl 942
full/euler/data/raw/normalized/gdelt/euler_2023-03-13.jsonl 65259
full/euler/data/raw/normalized/logs/euler_2023-03-13.jsonl 12993421
full/euler/data/raw/normalized/receipts/euler_2023-03-13.jsonl 1183299
full/euler/data/raw/normalized/token_transfers/euler_2023-03-13.jsonl 2068985663
full/euler/data/raw/normalized/traces/euler_2023-03-13.jsonl 42857553
full/euler/data/raw/normalized/transaction_index/euler_2023-03-13.jsonl 580554
full/euler/data/raw/normalized/transaction